# Compute and plot horizontal state-metric maps

Process surface variables or selected levels of 3-D variables, cache the requested diagnostics, and plot latitude–longitude maps.

## 1. Libraries


In [ ]:
from pathlib import Path
import sys
import xarray as xr
import matplotlib.pyplot as plt


## 2. User setup: paths


In [ ]:
# =============================================================================
# TOP-LEVEL PATH PARAMETERS
# Edit these three paths to relocate inputs, the analysis checkout, or outputs.
# =============================================================================
from pathlib import Path
import sys

INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")
WORKFLOW_NAME = "analysis_atm_init"


## 3. Project setup


In [ ]:
# Derived paths: built from the parameters above.
import sys
from pathlib import Path

DIAGNOSTIC_DIR = WORK_DIR / "diagnostic"
if not DIAGNOSTIC_DIR.is_dir():
    raise FileNotFoundError(f"Diagnostic work directory not found: {DIAGNOSTIC_DIR}")
if str(DIAGNOSTIC_DIR) not in sys.path:
    sys.path.insert(0, str(DIAGNOSTIC_DIR))

from configs.output_paths import workflow_output_dirs
WORKFLOW_DATA_DIR, WORKFLOW_FIGURE_DIR = workflow_output_dirs(
    WORKFLOW_NAME,
    output_root=DIAGNOSTIC_OUTPUT_ROOT,
)


## Notes: horizontal-map variables and metrics

Available surface variable:

| Variable | Description |
|---|---|
| `PS` | Surface pressure |

Available 3-D variables (a `level` must be selected):

| Variable | Description |
|---|---|
| `U` | Zonal wind |
| `V` | Meridional wind |
| `T` | Air temperature |
| `Q` | Specific humidity |
| `CLDLIQ` | Cloud liquid-water mixing ratio |
| `CLDICE` | Cloud ice mixing ratio |

Available metrics:

| Metric | Meaning |
|---|---|
| `bias` | Posterior minus the selected reference |
| `rmse` | Pointwise root-square posterior error against the reference |
| `spread` | Posterior ensemble standard deviation |

Each `MAP_REQUESTS` item selects a variable and metric. Add `"level": 500` (or
another available pressure/model level) for a 3-D variable; omit `level` for a
surface variable. Request-keyed NetCDF caches are reused unless
`force_compute = True`.


## 4. User setup: diagnostic selections


In [ ]:
ANALYSIS_DATA_DIR = WORKFLOW_DATA_DIR / "remapped_ensemble_diagnostics"
CACHE_DIR = WORKFLOW_DATA_DIR / "atmosphere_metrics"
FIGURE_DIR = WORKFLOW_FIGURE_DIR / "horizontal_maps"

EXPERIMENTS = ["CAM6-S0", "DART10-S0", "DART20-S0", "DART40-S0"]
REFERENCE_EXPERIMENT = "CAM6-S0"
TIMESTAMPS = [
    "2011-12-26-00000",
    "2011-12-26-21600",
    "2011-12-26-43200",
    "2011-12-26-64800",
]

MAP_REQUESTS = [
    {"variable": "PS", "metric": "bias"},
    {"variable": "PS", "metric": "spread"},
    # Example pressure-level request:
    # {"variable": "T", "metric": "rmse", "level": 500},
]

LAT_RANGE = (-90, 90)
LON_RANGE = (-180, 180)
force_compute = False
RUN_WORKFLOW = True
MAP_FIGSIZE = (10, 12)

SURFACE_VARIABLES = {"PS"}
THREE_D_VARIABLES = {"U", "V", "T", "Q", "CLDLIQ", "CLDICE"}
MAP_METRICS = {"bias", "rmse", "spread"}

MODEL_LABELS = {
    "CAM6-S0": "CAM6-DART (ENS=80)",
    "CTRL10-S0": "CTRL (ENS=10)",
    "DART10-S0": "EAM-DART (ENS=10)",
    "DART20-S0": "EAM-DART (ENS=20)",
    "DART40-S0": "EAM-DART (ENS=40)",
    "DART40INF0p6-S0": "EAM-DART (ENS=40, INF=0.6)",
}

FIGURE_DPI = 200


## 5. Workflow imports and availability


In [ ]:
import matplotlib.pyplot as plt
import xarray as xr

from util.atm_initialization_metrics import (
    DartAssimOutput,
    cache_map_metric,
    map_cache_path,
    plot_cached_maps,
)


## 6. Read, process, and cache


In [ ]:
if RUN_WORKFLOW:
    if REFERENCE_EXPERIMENT not in EXPERIMENTS:
        raise ValueError("REFERENCE_EXPERIMENT must be included in EXPERIMENTS")
    for request in MAP_REQUESTS:
        variable = request["variable"]
        metric = request["metric"]
        level = request.get("level")
        if variable not in SURFACE_VARIABLES | THREE_D_VARIABLES:
            raise ValueError(f"Unsupported variable: {variable}")
        if metric not in MAP_METRICS:
            raise ValueError(f"Unsupported horizontal-map metric: {metric}")
        if variable in THREE_D_VARIABLES and level is None:
            raise ValueError(f"A level is required for 3-D variable {variable}")
        if variable in SURFACE_VARIABLES and level is not None:
            raise ValueError(f"Surface variable {variable} must not specify a level")

    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    for timestamp in TIMESTAMPS:
        paths = {
            (experiment, request["variable"], request["metric"], request.get("level")):
                map_cache_path(
                    CACHE_DIR, experiment, timestamp, request["variable"],
                    request["metric"], request.get("level")
                )
            for experiment in EXPERIMENTS
            for request in MAP_REQUESTS
        }
        if not force_compute and all(path.exists() for path in paths.values()):
            print(f"{timestamp}: all horizontal metrics exist; reusing cache.")
            continue

        print(f"{timestamp}: deriving missing/forced horizontal metrics.")
        outputs = {
            experiment: DartAssimOutput.from_experiment(
                ANALYSIS_DATA_DIR, experiment, timestamp
            )
            for experiment in EXPERIMENTS
        }
        reference = outputs[REFERENCE_EXPERIMENT].post_mean
        try:
            for (experiment, variable, metric, level), path in paths.items():
                dataset = cache_map_metric(
                    path, outputs[experiment], reference, variable, metric,
                    LAT_RANGE, LON_RANGE, level=level,
                    force_compute=force_compute,
                )
                dataset.close()
        finally:
            for output in outputs.values():
                for dataset in (
                    output.prior_mean, output.prior_sd,
                    output.post_mean, output.post_sd,
                ):
                    dataset.close()


## 7. Plot, save, and show


## 4. Run the workflow and show results
